# Small illustrative example of NSGA-II for low-CO2 concrete beam design

The purpose of this notebook is to explain, in a simple and transparent way, how a multi-objective optimization procedure can be used to identify concrete beam solutions with low CO2 emissions and acceptable structural performance.

Instead of starting with the full database of beam sections, this notebook uses a very small set of candidate solutions. This makes it possible to inspect every solution manually and understand the logic of the method.

The notebook is structured in four stages:

1. Define a small set of beam solutions  
2. Check which solutions are structurally feasible  
3. Compare the solutions in terms of cost and CO2  
4. Introduce the logic of Pareto optimality and NSGA-II  

This notebook is intended as a teaching example before applying the same ideas to the full design space.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("ggplot")

## Step 1: Define a very small design space

To explain the optimization procedure clearly, we first create a small number of candidate beam solutions.

Each row in the table represents one beam design alternative.  
For each solution, we include:

- section width  
- section height  
- reinforcement area  
- bending resistance  
- required design moment  
- total CO2 per meter  
- total cost per meter  

The dataset is intentionally very small so that the behaviour of each solution can be followed easily.

In [3]:
example_df = pd.DataFrame({
    "Solution ID": ["A", "B", "C", "D", "E", "F", "G", "H"],
    "b [mm]": [200, 220, 240, 200, 250, 220, 260, 240],
    "h [mm]": [300, 320, 340, 360, 300, 380, 360, 400],
    "As [mm2]": [804, 804, 1005, 1257, 1005, 1257, 804, 1257],
    "M_Rd [kNm]": [55, 68, 82, 95, 72, 105, 78, 110],
    "M_Ed [kNm]": [76, 76, 76, 76, 76, 76, 76, 76],
    "Total CO2 [kg/m]": [20.5, 22.1, 24.6, 27.2, 23.0, 29.4, 25.5, 31.1],
    "Total cost [SEK/m]": [130, 138, 149, 162, 145, 171, 152, 178]
})

example_df

,Solution ID,b [mm],h [mm],As [mm2],M_Rd [kNm],M_Ed [kNm],Total CO2 [kg/m],Total cost [SEK/m]
0,A,200,300,804,55,76,20.5,130
1,B,220,320,804,68,76,22.1,138
2,C,240,340,1005,82,76,24.6,149
3,D,200,360,1257,95,76,27.2,162
4,E,250,300,1005,72,76,23.0,145
5,F,220,380,1257,105,76,29.4,171
6,G,260,360,804,78,76,25.5,152
7,H,240,400,1257,110,76,31.1,178


## Step 2: Check structural feasibility

Before comparing solutions in terms of sustainability and cost, we must first check whether each beam is structurally acceptable.

In this simplified example, a beam is considered feasible if its bending resistance is greater than or equal to the required design moment:

\[
M_{Rd} \geq M_{Ed}
\]

This is a very important step. A beam with very low CO2 is not useful if it cannot carry the applied load.

In [4]:
example_df["Feasible"] = example_df["M_Rd [kNm]"] >= example_df["M_Ed [kNm]"]

example_df[["Solution ID", "M_Rd [kNm]", "M_Ed [kNm]", "Feasible"]]

,Solution ID,M_Rd [kNm],M_Ed [kNm],Feasible
0,A,55,76,False
1,B,68,76,False
2,C,82,76,True
3,D,95,76,True
4,E,72,76,False
5,F,105,76,True
6,G,78,76,True
7,H,110,76,True


## Step 3: Separate feasible and non-feasible solutions

Now that feasibility has been evaluated, we split the design space into two groups:

- feasible solutions  
- non-feasible solutions  

This makes it easier to compare the two groups visually and understand how structural constraints influence the search for better designs.


In [ ]:

feasible_df = example_df[example_df["Feasible"]].copy()
nonfeasible_df = example_df[~example_df["Feasible"]].copy()

print("Number of feasible solutions:", len(feasible_df))
print("Number of non-feasible solutions:", len(nonfeasible_df))

## Step 4: Plot the design space in terms of CO2 and cost

We now represent the candidate beam solutions in the objective space.

The horizontal axis shows total CO2 per meter.  
The vertical axis shows total cost per meter.

This is the same type of plot that is commonly used in multi-objective optimization:
- each point is a design solution
- the lower-left region is desirable because it corresponds to both lower CO2 and lower cost

Feasible and non-feasible solutions are plotted with different colors so that the structural constraint remains visible.

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    feasible_df["Total CO2 [kg/m]"],
    feasible_df["Total cost [SEK/m]"],
    s=80,
    alpha=0.8,
    label="Feasible solutions"
)

plt.scatter(
    nonfeasible_df["Total CO2 [kg/m]"],
    nonfeasible_df["Total cost [SEK/m]"],
    s=80,
    alpha=0.8,
    label="Non-feasible solutions"
)

for _, row in example_df.iterrows():
    plt.annotate(
        row["Solution ID"],
        (row["Total CO2 [kg/m]"], row["Total cost [SEK/m]"]),
        xytext=(5, 5),
        textcoords="offset points"
    )

plt.xlabel("Total CO2 [kg/m]")
plt.ylabel("Total cost [SEK/m]")
plt.title("Small design space for beam solutions")
plt.legend()
plt.grid(True)
plt.show()

## Step 5: Introduce Pareto optimality

In a multi-objective problem, there is usually no single best solution.

Instead, we look for solutions that are **non-dominated**.

A solution is said to dominate another solution if:
- it is no worse in all objectives
- and strictly better in at least one objective

In this example, the objectives are:
- minimise total CO2
- minimise total cost

The set of non-dominated solutions is called the **Pareto front**.

These solutions represent the best trade-offs currently available in the design space.

## Step 6: Identify Pareto-efficient feasible solutions

We now identify which of the feasible solutions belong to the Pareto front.

For this small example, this can still be done computationally in a very transparent way.  
Later, NSGA-II will use this same idea repeatedly while generating and improving solutions over many generations.

In [ ]:
def is_dominated(row, df, obj_cols):
    for _, other in df.iterrows():
        if all(other[col] <= row[col] for col in obj_cols) and any(other[col] < row[col] for col in obj_cols):
            return True
    return False

objective_cols = ["Total CO2 [kg/m]", "Total cost [SEK/m]"]

feasible_df["Dominated"] = feasible_df.apply(
    lambda row: is_dominated(row, feasible_df, objective_cols), axis=1
)

feasible_df["Pareto optimal"] = ~feasible_df["Dominated"]

feasible_df[["Solution ID", "Total CO2 [kg/m]", "Total cost [SEK/m]", "Pareto optimal"]]

## Step 7: Visualise the Pareto-optimal feasible solutions

This plot highlights which feasible solutions are Pareto-optimal.

These are the solutions that NSGA-II would try to preserve and use as promising candidates during the optimization process.

At this stage, we are still not generating new designs.  
We are only learning how to evaluate and rank a small set of existing candidates.

In [ ]:
pareto_df = feasible_df[feasible_df["Pareto optimal"]].copy()

plt.figure(figsize=(8, 6))

plt.scatter(
    feasible_df["Total CO2 [kg/m]"],
    feasible_df["Total cost [SEK/m]"],
    s=80,
    alpha=0.5,
    label="Feasible solutions"
)

plt.scatter(
    pareto_df["Total CO2 [kg/m]"],
    pareto_df["Total cost [SEK/m]"],
    s=140,
    marker="D",
    edgecolors="black",
    linewidths=1.2,
    label="Pareto-optimal feasible solutions"
)

for _, row in feasible_df.iterrows():
    plt.annotate(
        row["Solution ID"],
        (row["Total CO2 [kg/m]"], row["Total cost [SEK/m]"]),
        xytext=(5, 5),
        textcoords="offset points"
    )

plt.xlabel("Total CO2 [kg/m]")
plt.ylabel("Total cost [SEK/m]")
plt.title("Pareto-optimal feasible beam solutions")
plt.legend()
plt.grid(True)
plt.show()

## Step 8: Connect this small example to NSGA-II

The small example above contains the core ideas behind NSGA-II:

- evaluate candidate solutions  
- reject or penalise non-feasible designs  
- compare solutions using multiple objectives  
- identify non-dominated solutions  
- preserve promising trade-offs  

In a real optimization problem, the number of possible beam designs is far too large to inspect manually.  
For that reason, NSGA-II is used to automate the search process.

In the next stage, the algorithm will:
1. start from an initial population of solutions  
2. rank them by non-domination  
3. preserve diversity among good solutions  
4. generate new candidate designs  
5. repeat the process over many generations  

This allows the search to move gradually toward beam designs with lower CO2 and acceptable structural performance.

## Step 9: Mimic one simple NSGA-II generation

NSGA-II is an evolutionary optimization algorithm.  
This means that it gradually improves a population of candidate solutions over several generations.

To explain the idea clearly, we now perform one simplified generation manually.

In this small example, we will:
- select two parent solutions
- combine their properties to form new child solutions
- apply a small mutation
- evaluate the children in the same way as the original designs

This is not yet a full NSGA-II implementation, but it demonstrates the main logic behind evolutionary optimization.

## Step 10: Select parent solutions

In an evolutionary algorithm, new solutions are created from existing ones.

Here, we select two parent beam designs from the feasible set.  
For teaching purposes, we choose them directly and visibly rather than using a probabilistic selection rule.

Later, in a more complete NSGA-II procedure, parent selection would be based on:
- non-dominated rank
- diversity in the population

In [ ]:
parent_1 = feasible_df.iloc[0].copy()
parent_2 = feasible_df.iloc[1].copy()

print("Parent 1")
display(parent_1)

print("Parent 2")
display(parent_2)

## Step 11: Create child solutions by crossover

Crossover means that the child solutions inherit properties from both parents.

In this example, we use a very simple crossover rule:
- one child takes the width from one parent and the height from the other
- reinforcement area is also mixed between the parents

This simplified step helps illustrate the principle that new designs can be generated by combining features of previously successful solutions.

In [ ]:
children_df = pd.DataFrame({
    "Solution ID": ["Child 1", "Child 2"],
    "b [mm]": [parent_1["b [mm]"], parent_2["b [mm]"]],
    "h [mm]": [parent_2["h [mm]"], parent_1["h [mm]"]],
    "As [mm2]": [parent_2["As [mm2]"], parent_1["As [mm2]"]]
})

children_df

## Step 12: Apply a small mutation

Mutation introduces variation into the population.

Without mutation, the algorithm would only recombine existing designs and could easily become trapped in a small part of the design space.

In this simple example, we apply a small manual mutation by changing one design variable slightly.  
This allows us to demonstrate how a new design can appear that is not identical to either parent.

In [ ]:
children_df.loc[0, "h [mm]"] += 20
children_df.loc[1, "b [mm]"] += 20

children_df

## Step 13: Define simple evaluation formulas for the child solutions

The child solutions must now be evaluated in the same way as the original candidate designs.

To keep the example transparent, we use simple approximate formulas:
- bending resistance increases with section size and reinforcement area
- CO2 increases with section size and reinforcement area
- cost also increases with section size and reinforcement area

These formulas are not intended to replace the detailed calculations in the main notebook.  
Their purpose is only to illustrate the optimization logic in a compact teaching example.

In [ ]:
def evaluate_children(df, design_moment=76):
    df = df.copy()

    # Simple illustrative formulas
    df["M_Rd [kNm]"] = 0.00055 * df["b [mm]"] * df["h [mm]"] + 0.018 * df["As [mm2]"]
    df["M_Ed [kNm]"] = design_moment

    df["Total CO2 [kg/m]"] = (
        0.00018 * df["b [mm]"] * df["h [mm]"] +
        0.0025 * df["As [mm2]"]
    )

    df["Total cost [SEK/m]"] = (
        0.0010 * df["b [mm]"] * df["h [mm]"] +
        0.080 * df["As [mm2]"]
    )

    df["Feasible"] = df["M_Rd [kNm]"] >= df["M_Ed [kNm]"]

    return df

children_df = evaluate_children(children_df)
children_df

## Step 14: Compare parents and children in one table

We now collect the selected parent solutions and the new child solutions in one combined table.

This allows us to compare:
- geometry
- reinforcement
- structural resistance
- feasibility
- CO2
- cost

This is the same general idea used in evolutionary optimization:  
an old population and a new population are compared, and the most promising solutions are kept for the next generation.

In [ ]:
parents_df = pd.DataFrame([parent_1, parent_2])[[
    "Solution ID", "b [mm]", "h [mm]", "As [mm2]",
    "M_Rd [kNm]", "M_Ed [kNm]", "Total CO2 [kg/m]", "Total cost [SEK/m]", "Feasible"
]].copy()

combined_generation_df = pd.concat([
    parents_df,
    children_df[[
        "Solution ID", "b [mm]", "h [mm]", "As [mm2]",
        "M_Rd [kNm]", "M_Ed [kNm]", "Total CO2 [kg/m]", "Total cost [SEK/m]", "Feasible"
    ]]
], ignore_index=True)

combined_generation_df

## Step 15: Plot parents and children in the objective space

This plot shows how the new child solutions compare with the selected parent solutions.

The objective space remains the same:
- horizontal axis: total CO2
- vertical axis: total cost

This helps us see whether the new generation has moved toward more attractive regions of the design space.

In a real NSGA-II procedure, many solutions would be evaluated in each generation, and this improvement process would continue over many iterations.

In [ ]:
plt.figure(figsize=(8, 6))

parent_points = combined_generation_df[combined_generation_df["Solution ID"].isin(["A", "B"])]
child_points = combined_generation_df[combined_generation_df["Solution ID"].isin(["Child 1", "Child 2"])]

plt.scatter(
    parent_points["Total CO2 [kg/m]"],
    parent_points["Total cost [SEK/m]"],
    s=140,
    marker="o",
    alpha=0.8,
    label="Parents"
)

plt.scatter(
    child_points["Total CO2 [kg/m]"],
    child_points["Total cost [SEK/m]"],
    s=140,
    marker="D",
    alpha=0.8,
    label="Children"
)

for _, row in combined_generation_df.iterrows():
    plt.annotate(
        row["Solution ID"],
        (row["Total CO2 [kg/m]"], row["Total cost [SEK/m]"]),
        xytext=(5, 5),
        textcoords="offset points"
    )

plt.xlabel("Total CO2 [kg/m]")
plt.ylabel("Total cost [SEK/m]")
plt.title("One simplified evolutionary step")
plt.legend()
plt.grid(True)
plt.show()

## Step 16: Explain how this connects to NSGA-II

The simplified generation above already contains the most important ingredients of an evolutionary optimization method:

- parent solutions are selected  
- children are created through crossover  
- mutation introduces variation  
- new solutions are evaluated  
- promising solutions can be retained  

NSGA-II extends this idea in a more systematic way.

In the full algorithm:
- many candidate solutions are handled at once
- solutions are ranked by non-domination
- diversity is preserved using crowding distance
- the process is repeated for many generations

The result is a population that moves toward better trade-offs between structural feasibility, low CO2, and low cost.

## Step 17: Rank the combined population using non-dominated sorting

After generating child solutions, we now have a combined population containing both parents and children.

The next task is to determine which solutions are more attractive in a multi-objective sense.

In NSGA-II, this is done by **non-dominated sorting**.

The idea is:
- solutions that are not dominated by any other solution belong to the first Pareto front
- after removing these, the next set of non-dominated solutions forms the second front
- this process continues until all solutions have been ranked

In this teaching example, we use only the first front to illustrate the principle clearly.

## Step 18: Define a simple dominance rule with feasibility

In beam design, structural feasibility must always come before optimization of cost and CO2.

For that reason, we use the following logic:
- a feasible solution is always preferred over a non-feasible solution
- between two feasible solutions, Pareto dominance is checked using CO2 and cost
- between two non-feasible solutions, Pareto dominance can still be checked, but they remain less attractive than feasible ones

This reflects the real design problem: a low-CO2 beam is only interesting if it can actually carry the required load.

In [ ]:
objective_cols = ["Total CO2 [kg/m]", "Total cost [SEK/m]"]

def dominates(row_a, row_b, obj_cols):
    # Feasible solutions are always preferred over non-feasible ones
    if row_a["Feasible"] and not row_b["Feasible"]:
        return True
    if not row_a["Feasible"] and row_b["Feasible"]:
        return False

    # If both have same feasibility status, compare objectives
    no_worse = all(row_a[col] <= row_b[col] for col in obj_cols)
    strictly_better = any(row_a[col] < row_b[col] for col in obj_cols)

    return no_worse and strictly_better

## Step 19: Identify the first Pareto front

We now check each solution in the combined population and determine whether it is dominated by any other solution.

If a solution is not dominated by any other solution, it belongs to the first Pareto front.

In NSGA-II, these first-front solutions are especially important because they represent the best currently available trade-offs between the objectives.

In [ ]:
def get_first_pareto_front(df, obj_cols):
    front_indices = []

    for i, row_i in df.iterrows():
        dominated_flag = False

        for j, row_j in df.iterrows():
            if i == j:
                continue

            if dominates(row_j, row_i, obj_cols):
                dominated_flag = True
                break

        if not dominated_flag:
            front_indices.append(i)

    return df.loc[front_indices].copy()

first_front_df = get_first_pareto_front(combined_generation_df, objective_cols)
first_front_df

## Step 20: Visualise the first Pareto front

This figure highlights the solutions in the first Pareto front.

These are the solutions that would be most attractive for survival into the next generation.

Even in this very small example, we can already see an important idea of multi-objective optimization:
there is often not one single best solution, but rather a set of competing trade-off solutions.

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    combined_generation_df["Total CO2 [kg/m]"],
    combined_generation_df["Total cost [SEK/m]"],
    s=100,
    alpha=0.4,
    label="Combined population"
)

plt.scatter(
    first_front_df["Total CO2 [kg/m]"],
    first_front_df["Total cost [SEK/m]"],
    s=180,
    marker="D",
    edgecolors="black",
    linewidths=1.2,
    label="First Pareto front"
)

for _, row in combined_generation_df.iterrows():
    plt.annotate(
        row["Solution ID"],
        (row["Total CO2 [kg/m]"], row["Total cost [SEK/m]"]),
        xytext=(5, 5),
        textcoords="offset points"
    )

plt.xlabel("Total CO2 [kg/m]")
plt.ylabel("Total cost [SEK/m]")
plt.title("First Pareto front in the combined population")
plt.legend()
plt.grid(True)
plt.show()

## Step 21: Assign a simple Pareto rank

To prepare for next-generation selection, we now assign a simple rank to each solution.

For this teaching example:
- solutions in the first Pareto front receive rank 1
- all other solutions receive rank 2

In a full NSGA-II implementation, this would continue with rank 3, rank 4, and so on until all solutions are classified.

In [ ]:
combined_generation_df = combined_generation_df.copy()
combined_generation_df["Pareto rank"] = 2
combined_generation_df.loc[first_front_df.index, "Pareto rank"] = 1

combined_generation_df[[
    "Solution ID", "Feasible", "Total CO2 [kg/m]", "Total cost [SEK/m]", "Pareto rank"
]]

## Step 22: Select the next generation

The purpose of NSGA-II is not only to generate new solutions, but also to decide which solutions should survive into the next generation.

In this simple example, we choose a small population size and keep the best solutions according to:
1. feasibility  
2. Pareto rank  
3. objective values  

This is still a simplified version of NSGA-II, but it already shows the central idea of survival selection.

In [ ]:
next_generation_size = 4

next_generation_df = combined_generation_df.sort_values(
    by=["Feasible", "Pareto rank", "Total CO2 [kg/m]", "Total cost [SEK/m]"],
    ascending=[False, True, True, True]
).head(next_generation_size).copy()

next_generation_df

## Step 23: Interpret the selected next generation

The table above represents the surviving solutions after one simplified generation.

These are the beam designs that would be used as the basis for the next round of optimization.

This reflects the main evolutionary idea:
- weaker solutions disappear
- stronger solutions survive
- new solutions are generated from the survivors

By repeating this procedure many times, the population can gradually move toward better and better feasible low-CO2 solutions.

## Step 24: Plot the selected next generation

This final plot highlights the solutions that survive into the next generation.

The aim is to make the selection process easy to understand visually:
- the full combined population is shown in the background
- the selected survivors are highlighted

This closes one complete simplified optimization cycle:
- evaluate
- rank
- generate children
- compare populations
- keep the best solutions

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    combined_generation_df["Total CO2 [kg/m]"],
    combined_generation_df["Total cost [SEK/m]"],
    s=100,
    alpha=0.35,
    label="Combined population"
)

plt.scatter(
    next_generation_df["Total CO2 [kg/m]"],
    next_generation_df["Total cost [SEK/m]"],
    s=180,
    marker="*",
    edgecolors="black",
    linewidths=1.2,
    label="Selected next generation"
)

for _, row in combined_generation_df.iterrows():
    plt.annotate(
        row["Solution ID"],
        (row["Total CO2 [kg/m]"], row["Total cost [SEK/m]"]),
        xytext=(5, 5),
        textcoords="offset points"
    )

plt.xlabel("Total CO2 [kg/m]")
plt.ylabel("Total cost [SEK/m]")
plt.title("Selection of the next generation")
plt.legend()
plt.grid(True)
plt.show()

## Step 25: Summary of the simplified NSGA-II logic

At this point, the notebook has demonstrated the main ideas behind NSGA-II in a small and transparent way.

The procedure can now be summarized as follows:

1. Define candidate beam solutions  
2. Check structural feasibility  
3. Evaluate cost and CO2  
4. Identify non-dominated solutions  
5. Generate child solutions from good parents  
6. Combine old and new populations  
7. Select the best solutions for the next generation  

In the full NSGA-II algorithm, this process is repeated automatically for many generations, and diversity between solutions is also preserved using the crowding-distance concept.

This small example is therefore a useful bridge between manual engineering comparison and full multi-objective evolutionary optimization.

## Step 26: Move from one generation to an iterative optimization loop

So far, the notebook has shown one simplified evolutionary step.

However, the main strength of NSGA-II is that the same logic is repeated over many generations.  
This allows the search process to gradually move toward better trade-offs between structural feasibility, low CO2, and low cost.

In this section, we implement a small iterative loop that:
- starts from an initial population
- evaluates all solutions
- selects promising designs
- generates children
- repeats the procedure for several generations

The purpose is not yet to create a full advanced NSGA-II solver, but rather to demonstrate the core iterative mechanism in a clear way.

## Step 26: Move from one generation to an iterative optimization loop

So far, the notebook has shown one simplified evolutionary step.

However, the main strength of NSGA-II is that the same logic is repeated over many generations.  
This allows the search process to gradually move toward better trade-offs between structural feasibility, low CO2, and low cost.

In this section, we implement a small iterative loop that:
- starts from an initial population
- evaluates all solutions
- selects promising designs
- generates children
- repeats the procedure for several generations

The purpose is not yet to create a full advanced NSGA-II solver, but rather to demonstrate the core iterative mechanism in a clear way.

In [ ]:
objective_cols = ["Total CO2 [kg/m]", "Total cost [SEK/m]"]

def evaluate_population(df, design_moment=76):
    df = df.copy()

    # Simple illustrative formulas
    df["M_Rd [kNm]"] = 0.00055 * df["b [mm]"] * df["h [mm]"] + 0.018 * df["As [mm2]"]
    df["M_Ed [kNm]"] = design_moment

    df["Total CO2 [kg/m]"] = (
        0.00018 * df["b [mm]"] * df["h [mm]"] +
        0.0025 * df["As [mm2]"]
    )

    df["Total cost [SEK/m]"] = (
        0.0010 * df["b [mm]"] * df["h [mm]"] +
        0.080 * df["As [mm2]"]
    )

    df["Feasible"] = df["M_Rd [kNm]"] >= df["M_Ed [kNm]"]

    return df


def dominates(row_a, row_b, obj_cols):
    if row_a["Feasible"] and not row_b["Feasible"]:
        return True
    if not row_a["Feasible"] and row_b["Feasible"]:
        return False

    no_worse = all(row_a[col] <= row_b[col] for col in obj_cols)
    strictly_better = any(row_a[col] < row_b[col] for col in obj_cols)

    return no_worse and strictly_better


def get_first_pareto_front(df, obj_cols):
    front_indices = []

    for i, row_i in df.iterrows():
        dominated_flag = False
        for j, row_j in df.iterrows():
            if i == j:
                continue
            if dominates(row_j, row_i, obj_cols):
                dominated_flag = True
                break
        if not dominated_flag:
            front_indices.append(i)

    return df.loc[front_indices].copy()


def make_child(parent_1, parent_2, child_name):
    child = {}

    child["Solution ID"] = child_name

    # Simple crossover
    child["b [mm]"] = np.random.choice([parent_1["b [mm]"], parent_2["b [mm]"]])
    child["h [mm]"] = np.random.choice([parent_1["h [mm]"], parent_2["h [mm]"]])
    child["As [mm2]"] = np.random.choice([parent_1["As [mm2]"], parent_2["As [mm2]"]])

    # Simple mutation
    if np.random.rand() < 0.5:
        child["b [mm]"] += np.random.choice([-20, 20])
    if np.random.rand() < 0.5:
        child["h [mm]"] += np.random.choice([-20, 20])
    if np.random.rand() < 0.5:
        child["As [mm2]"] += np.random.choice([-201, 0, 201])

    # Keep values within simple bounds
    child["b [mm]"] = int(np.clip(child["b [mm]"], 180, 300))
    child["h [mm]"] = int(np.clip(child["h [mm]"], 280, 450))
    child["As [mm2]"] = int(np.clip(child["As [mm2]"], 603, 1500))

    return child

## Step 28: Define the initial population

We now define the starting population for the iterative search.

For this teaching example, the initial population is simply the small set of candidate beam solutions introduced earlier.

In a more advanced implementation, the initial population could be generated randomly within specified design-variable bounds.

In [ ]:
population_df = example_df[["Solution ID", "b [mm]", "h [mm]", "As [mm2]"]].copy()
population_df = evaluate_population(population_df)

population_df

## Step 29: Run the simplified evolutionary loop

The loop below performs several generations of evolutionary search.

In each generation, it does the following:
1. evaluate the current population  
2. identify the first Pareto front  
3. select parent solutions  
4. create children through crossover and mutation  
5. combine parents and children  
6. keep the best solutions for the next generation  

This repeats the same core logic that was previously shown for just one generation, but now in an automated way.

In [ ]:
np.random.seed(42)

population_size = 8
n_generations = 10

history = []

current_population = population_df.copy()

for generation in range(n_generations):
    current_population = evaluate_population(current_population)

    first_front = get_first_pareto_front(current_population, objective_cols).copy()
    first_front["Generation"] = generation

    history.append(first_front)

    # Parent pool: prefer feasible solutions in first front
    feasible_front = first_front[first_front["Feasible"]].copy()

    if len(feasible_front) >= 2:
        parent_pool = feasible_front
    elif len(first_front) >= 2:
        parent_pool = first_front
    else:
        parent_pool = current_population.copy()

    # Create children
    children = []
    for child_idx in range(population_size):
        parents = parent_pool.sample(2, replace=True)
        parent_1 = parents.iloc[0]
        parent_2 = parents.iloc[1]

        child = make_child(parent_1, parent_2, f"G{generation+1}_C{child_idx+1}")
        children.append(child)

    children_df = pd.DataFrame(children)
    children_df = evaluate_population(children_df)

    # Combine current population and children
    combined_df = pd.concat([current_population, children_df], ignore_index=True)

    # Rank using only first front vs others
    combined_df["Pareto rank"] = 2
    combined_first_front = get_first_pareto_front(combined_df, objective_cols)
    combined_df.loc[combined_first_front.index, "Pareto rank"] = 1

    # Select next generation
    current_population = combined_df.sort_values(
        by=["Feasible", "Pareto rank", "Total CO2 [kg/m]", "Total cost [SEK/m]"],
        ascending=[False, True, True, True]
    ).head(population_size).copy()

current_population

## Step 30: Collect the Pareto-front history over all generations

To understand how the optimization develops over time, it is useful to store the Pareto front from each generation.

This makes it possible to observe whether the population is gradually moving toward lower CO2 and lower cost while maintaining structural feasibility.

In [ ]:
history_df = pd.concat(history, ignore_index=True)
history_df

## Step 31: Plot the Pareto-front solutions from all generations

This figure shows the Pareto-front solutions that appeared during the iterative search.

Each point represents a solution that belonged to the first Pareto front in at least one generation.

This helps us see how the optimization process explores the design space over time.

In [ ]:
plt.figure(figsize=(9, 6))

scatter = plt.scatter(
    history_df["Total CO2 [kg/m]"],
    history_df["Total cost [SEK/m]"],
    c=history_df["Generation"],
    s=100,
    alpha=0.8
)

for _, row in history_df.iterrows():
    plt.annotate(
        row["Solution ID"],
        (row["Total CO2 [kg/m]"], row["Total cost [SEK/m]"]),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=8
    )

plt.xlabel("Total CO2 [kg/m]")
plt.ylabel("Total cost [SEK/m]")
plt.title("Pareto-front solutions over successive generations")
plt.grid(True)
plt.colorbar(scatter, label="Generation")
plt.show()

## Step 32: Show the final population

After the loop ends, the final population represents the surviving solutions after several rounds of selection, crossover, mutation, and survival.

These solutions are not guaranteed to be globally optimal, but they illustrate the main purpose of NSGA-II:
to move the search toward better feasible trade-offs between competing objectives.

In [ ]:
final_population = current_population.copy()
final_population

## Step 33: Highlight the final feasible Pareto front

The final step is to identify the Pareto-efficient feasible solutions in the last generation.

These are the solutions that represent the best trade-offs found by the simplified iterative search.

In [ ]:
final_front = get_first_pareto_front(final_population, objective_cols).copy()

plt.figure(figsize=(9, 6))

plt.scatter(
    final_population["Total CO2 [kg/m]"],
    final_population["Total cost [SEK/m]"],
    s=100,
    alpha=0.4,
    label="Final population"
)

plt.scatter(
    final_front["Total CO2 [kg/m]"],
    final_front["Total cost [SEK/m]"],
    s=180,
    marker="D",
    edgecolors="black",
    linewidths=1.2,
    label="Final Pareto front"
)

for _, row in final_population.iterrows():
    plt.annotate(
        row["Solution ID"],
        (row["Total CO2 [kg/m]"], row["Total cost [SEK/m]"]),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=8
    )

plt.xlabel("Total CO2 [kg/m]")
plt.ylabel("Total cost [SEK/m]")
plt.title("Final population and final Pareto front")
plt.legend()
plt.grid(True)
plt.show()

final_front

## Step 34: Interpretation of the iterative loop

The iterative loop above demonstrates the essential optimization mechanism behind NSGA-II.

Over successive generations:
- candidate beam solutions are evaluated
- promising solutions are preserved
- variation is introduced through crossover and mutation
- the search gradually moves through the design space

In a full NSGA-II implementation, this procedure would be made more robust by:
- calculating full non-dominated ranks
- using crowding distance to maintain diversity
- applying more systematic parent selection
- handling design constraints more carefully

Even so, this teaching example already shows how an evolutionary multi-objective search can be used to identify beam designs with low CO2 and acceptable structural performance.

## Simple iterative NSGA-II style search

This section shows a small evolutionary search loop.

We begin with a small population of beam solutions and repeat the same steps over several generations:
- evaluate each solution
- identify the best trade-off solutions
- use them as parents
- create new children by crossover and mutation
- keep the most promising solutions

The purpose is to illustrate the logic of NSGA-II in a simple way before using a more advanced implementation on the full beam design space.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

objective_cols = ["Total CO2 [kg/m]", "Total cost [SEK/m]"]

def evaluate_population(df, design_moment=76):
    df = df.copy()

    # Simple illustrative formulas
    df["M_Rd [kNm]"] = 0.00055 * df["b [mm]"] * df["h [mm]"] + 0.018 * df["As [mm2]"]
    df["M_Ed [kNm]"] = design_moment

    df["Total CO2 [kg/m]"] = (
        0.00018 * df["b [mm]"] * df["h [mm]"] +
        0.0025 * df["As [mm2]"]
    )

    df["Total cost [SEK/m]"] = (
        0.0010 * df["b [mm]"] * df["h [mm]"] +
        0.080 * df["As [mm2]"]
    )

    df["Feasible"] = df["M_Rd [kNm]"] >= df["M_Ed [kNm]"]
    return df

def dominates(row_a, row_b, obj_cols):
    if row_a["Feasible"] and not row_b["Feasible"]:
        return True
    if not row_a["Feasible"] and row_b["Feasible"]:
        return False

    no_worse = all(row_a[col] <= row_b[col] for col in obj_cols)
    strictly_better = any(row_a[col] < row_b[col] for col in obj_cols)
    return no_worse and strictly_better

def get_first_pareto_front(df, obj_cols):
    front_indices = []

    for i, row_i in df.iterrows():
        dominated_flag = False
        for j, row_j in df.iterrows():
            if i == j:
                continue
            if dominates(row_j, row_i, obj_cols):
                dominated_flag = True
                break
        if not dominated_flag:
            front_indices.append(i)

    return df.loc[front_indices].copy()

def make_child(parent_1, parent_2, child_name):
    child = {}
    child["Solution ID"] = child_name

    # Crossover
    child["b [mm]"] = np.random.choice([parent_1["b [mm]"], parent_2["b [mm]"]])
    child["h [mm]"] = np.random.choice([parent_1["h [mm]"], parent_2["h [mm]"]])
    child["As [mm2]"] = np.random.choice([parent_1["As [mm2]"], parent_2["As [mm2]"]])

    # Mutation
    if np.random.rand() < 0.5:
        child["b [mm]"] += np.random.choice([-20, 20])
    if np.random.rand() < 0.5:
        child["h [mm]"] += np.random.choice([-20, 20])
    if np.random.rand() < 0.5:
        child["As [mm2]"] += np.random.choice([-201, 0, 201])

    # Bounds
    child["b [mm]"] = int(np.clip(child["b [mm]"], 180, 300))
    child["h [mm]"] = int(np.clip(child["h [mm]"], 280, 450))
    child["As [mm2]"] = int(np.clip(child["As [mm2]"], 603, 1500))

    return child

# Initial population
population_df = example_df[["Solution ID", "b [mm]", "h [mm]", "As [mm2]"]].copy()

population_size = 8
n_generations = 10
history = []

current_population = evaluate_population(population_df)

for generation in range(n_generations):
    first_front = get_first_pareto_front(current_population, objective_cols).copy()
    first_front["Generation"] = generation
    history.append(first_front)

    feasible_front = first_front[first_front["Feasible"]].copy()

    if len(feasible_front) >= 2:
        parent_pool = feasible_front
    elif len(first_front) >= 2:
        parent_pool = first_front
    else:
        parent_pool = current_population.copy()

    children = []
    for child_idx in range(population_size):
        parents = parent_pool.sample(2, replace=True)
        parent_1 = parents.iloc[0]
        parent_2 = parents.iloc[1]
        children.append(make_child(parent_1, parent_2, f"G{generation+1}_C{child_idx+1}"))

    children_df = pd.DataFrame(children)
    children_df = evaluate_population(children_df)

    combined_df = pd.concat([current_population, children_df], ignore_index=True)
    combined_df["Pareto rank"] = 2

    combined_front = get_first_pareto_front(combined_df, objective_cols)
    combined_df.loc[combined_front.index, "Pareto rank"] = 1

    current_population = combined_df.sort_values(
        by=["Feasible", "Pareto rank", "Total CO2 [kg/m]", "Total cost [SEK/m]"],
        ascending=[False, True, True, True]
    ).head(population_size).copy()

history_df = pd.concat(history, ignore_index=True)
final_population = current_population.copy()
final_front = get_first_pareto_front(final_population, objective_cols).copy()

plt.figure(figsize=(9, 6))
plt.scatter(
    history_df["Total CO2 [kg/m]"],
    history_df["Total cost [SEK/m]"],
    c=history_df["Generation"],
    s=100,
    alpha=0.8
)
plt.xlabel("Total CO2 [kg/m]")
plt.ylabel("Total cost [SEK/m]")
plt.title("Pareto-front solutions over successive generations")
plt.grid(True)
plt.colorbar(label="Generation")
plt.show()

display(final_population)
display(final_front)